# 08 — Seleção do método de clusterização e equalização de cardinalidade

Compara os quatro métodos de clusterização presentes no notebook de referência — K-Means com representante real (denominado `K-Medoids` na referência), MiniBatch K-Means, mistura Gaussiana (GMM) e clusterização hierárquica aglomerativa com ligação Ward — e inclui também o Farthest-Point Sampling (FPS).

A comparação usa as 90 frentes CNBI completas do FULL. O alvo de cada cenário–semente é a menor cardinalidade válida entre os métodos naquele bloco. Métodos estocásticos são avaliados com cinco inicializações. A escolha exige cardinalidade exata em todos os ensaios e minimiza, nesta ordem: posto mediano de IGD contra a fronteira verdadeira, IGD mediana, perda mediana de cobertura da frente CNBI original e tempo mediano.

Depois da escolha, todas as linhas de cardinalidade equalizada são recalculadas com o método vencedor e semente de redução fixa 42. As comparações visuais usam a semente comum 101 para preservar o pareamento entre métodos. Nenhum otimizador é reexecutado.

In [ ]:
from pathlib import Path
import json, math, os, re, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from scipy.spatial import cKDTree
from scipy.stats import qmc
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from statsmodels.stats.multitest import multipletests
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper(); ALPHA=2**0.75
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
TAB=ROOT/'results'/'tables'; FIG=ROOT/'results'/'figures'; RED_FIG=FIG/'cardinality_reduction'; EQ_FIG=FIG/'equalized_combined_projections'
RED_FIG.mkdir(parents=True,exist_ok=True); EQ_FIG.mkdir(parents=True,exist_ok=True)
CLUSTER_SEEDS=[1,2,3,4,5]; FINAL_REDUCTION_SEED=42; VISUAL_SEED=101; BENCH_HV_SAMPLES=8192
CLUSTER_METHODS=['Farthest-Point-Sampling','KMeans-medoid','MiniBatchKMeans-medoid','GMM-medoid','Hierarchical-Ward-medoid']
CLUSTER_LABEL={'Farthest-Point-Sampling':'Farthest-Point Sampling','KMeans-medoid':'K-Means + representante real','MiniBatchKMeans-medoid':'MiniBatch K-Means + representante real','GMM-medoid':'GMM + representante real','Hierarchical-Ward-medoid':'Hierárquica Ward + representante real'}
CLUSTER_COLOR={'Farthest-Point-Sampling':'#e45756','KMeans-medoid':'#4c78a8','MiniBatchKMeans-medoid':'#f58518','GMM-medoid':'#54a24b','Hierarchical-Ward-medoid':'#b279a2'}
METHOD_ORDER=['NBI','CNBI','VRF-NBI','NSGA-III','MOEA/D']
METHOD_COLOR={'NBI':'#0072b2','CNBI':'#d55e00','VRF-NBI':'#009e73','NSGA-III':'#cc79a7','MOEA/D':'#e69f00'}
METHOD_MARKER={'NBI':'P','CNBI':'o','VRF-NBI':'^','NSGA-III':'s','MOEA/D':'D'}
SELECTIONS={'m4_medium':{'pairs':[(1,2),(3,4)],'triples':[(1,2,3),(2,3,4)]},'m6_low':{'pairs':[(1,2),(3,4),(5,6)],'triples':[(1,2,3),(4,5,6)]},'m6_medium':{'pairs':[(1,2),(3,4),(5,6)],'triples':[(1,2,3),(4,5,6)]},'m12_high':{'pairs':[(1,2),(6,7),(11,12)],'triples':[(1,2,3),(5,6,7),(10,11,12)]}}
ISO_ELEV=35.264; ISO_AZIM=-45.0
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':9,'axes.titlesize':10,'axes.labelsize':9,'legend.fontsize':8,'figure.titlesize':13,'savefig.facecolor':'white','axes.facecolor':'white'})


In [ ]:
def nearest(A,B): return cKDTree(B).query(A,k=1)[0]
def GD(A,R): return float(nearest(A,R).mean())
def IGD(A,R): return float(nearest(R,A).mean())
def spacing(A):
    if len(A)<2: return 0.0
    d=cKDTree(A).query(A,k=2)[0][:,1]; return float(np.sqrt(np.mean((d-d.mean())**2)))
def sparsity(A):
    if len(A)<2: return 0.0
    return float(cKDTree(A).query(A,k=2)[0][:,1].mean())
def hv_from_samples(A,Z,ref=1.1):
    cap=24_000_000; chunk=max(500,min(20000,int(cap/max(len(A)*A.shape[1],1)))); hit=0
    for start in range(0,len(Z),chunk): hit+=np.any(np.all(Z[start:start+chunk,None,:]>=A[None,:,:],axis=2),axis=1).sum()
    p=hit/len(Z); volume=ref**A.shape[1]; return float(p*volume),float(volume*np.sqrt(max(p*(1-p),0)/len(Z)))

metrics_existing=pd.read_csv(TAB/f'{MODE.lower()}_metrics.csv'); complete=metrics_existing[metrics_existing.comparison.eq('complete')].copy()
runs=pd.read_csv(TAB/f'{MODE.lower()}_method_runs.csv'); records=complete.merge(runs[['scenario','seed','method','checkpoint']],on=['scenario','seed','method'],how='left',validate='one_to_one')
assert records.checkpoint.notna().all(); targets=complete.groupby(['scenario','seed']).n.min().astype(int).to_dict(); front_cache={}

def load_front(row):
    key=(row.scenario,int(row.seed),row.method)
    if key in front_cache: return front_cache[key]
    data=np.load(ROOT/row.checkpoint,allow_pickle=False); X=np.asarray(data['X'],float); success=np.asarray(data['success'],bool); data.close()
    feasible=np.sum(X*X,axis=1)<=ALPHA**2+1e-8; X=X[success&feasible]
    scenario_data=np.load(ROOT/'data'/'generated'/f'{row.scenario}_scenario.npz'); anchors=np.asarray(scenario_data['anchors'],float); scenario_data.close()
    Fraw=np.sum((X[:,None,:]-anchors[None,:,:])**2,axis=2); keep=NonDominatedSorting().do(Fraw,only_non_dominated_front=True); Fraw=Fraw[keep]
    reference=np.load(ROOT/'data'/'reference_fronts'/f'{row.scenario}_pareto_reference.npz'); Rraw=np.asarray(reference['F'],float); ideal=np.asarray(reference['ideal_true'],float); amp=np.maximum(np.asarray(reference['nadir_true'],float)-ideal,1e-12); reference.close()
    Fn=(Fraw-ideal)/amp; Rn=(Rraw-ideal)/amp; assert len(Fn)==int(row.n),f'Cardinalidade divergente: {key}'
    front_cache[key]=(Fraw,Fn,Rraw,Rn,anchors); return front_cache[key]


In [ ]:
def nearest_representatives(F,labels,centers,n):
    idx=[]
    for cluster in range(n):
        members=np.where(labels==cluster)[0]
        if len(members): idx.append(members[np.argmin(np.linalg.norm(F[members]-centers[cluster],axis=1))])
    return np.sort(np.unique(np.asarray(idx,dtype=int)))

def reduce_indices(F,n,method,seed=42):
    F=np.asarray(F,float)
    if n>=len(F): return np.arange(len(F),dtype=int)
    if method=='Farthest-Point-Sampling':
        rng=np.random.default_rng(seed); idx=[int(rng.integers(len(F)))]; distance=np.linalg.norm(F-F[idx[0]],axis=1)
        for _ in range(n-1):
            next_idx=int(np.argmax(distance)); idx.append(next_idx); distance=np.minimum(distance,np.linalg.norm(F-F[next_idx],axis=1))
        return np.sort(np.asarray(idx,dtype=int))
    if method=='KMeans-medoid':
        model=KMeans(n_clusters=n,random_state=seed,n_init=10).fit(F); return nearest_representatives(F,model.labels_,model.cluster_centers_,n)
    if method=='MiniBatchKMeans-medoid':
        model=MiniBatchKMeans(n_clusters=n,random_state=seed,batch_size=256,n_init=10).fit(F); return nearest_representatives(F,model.labels_,model.cluster_centers_,n)
    if method=='GMM-medoid':
        model=GaussianMixture(n_components=n,covariance_type='diag',random_state=seed,n_init=3,reg_covar=1e-6).fit(F); labels=model.predict(F); return nearest_representatives(F,labels,model.means_,n)
    if method=='Hierarchical-Ward-medoid':
        labels=AgglomerativeClustering(n_clusters=n,linkage='ward').fit_predict(F); centers=np.vstack([F[labels==cluster].mean(axis=0) for cluster in range(n)]); return nearest_representatives(F,labels,centers,n)
    raise KeyError(method)

def reduction_seeds(method): return [None] if method=='Hierarchical-Ward-medoid' else CLUSTER_SEEDS

bench=[]
for row in records[records.method.eq('CNBI')].sort_values(['scenario','seed']).itertuples(index=False):
    _,Fn,_,Rn,_=load_front(row); target=targets[(row.scenario,int(row.seed))]
    Z=qmc.Sobol(Fn.shape[1],scramble=True,seed=7000+int(row.seed)).random(BENCH_HV_SAMPLES)*1.1
    for cluster_method in CLUSTER_METHODS:
        for cluster_seed in reduction_seeds(cluster_method):
            started=time.perf_counter(); status='COMPLETED'
            try: idx=reduce_indices(Fn,target,cluster_method,42 if cluster_seed is None else cluster_seed)
            except Exception as exc: idx=np.array([],dtype=int); status=f'ERROR:{type(exc).__name__}'
            elapsed=time.perf_counter()-started; exact=len(idx)==target
            values={'retention_IGD':np.nan,'retention_p95':np.nan,'retention_max':np.nan,'GD_true':np.nan,'IGD_true':np.nan,'HV':np.nan,'HV_se':np.nan,'Spacing':np.nan,'Sparsity':np.nan}
            if exact:
                reduced=Fn[idx]; loss=nearest(Fn,reduced); hv,hse=hv_from_samples(reduced,Z)
                values={'retention_IGD':float(loss.mean()),'retention_p95':float(np.quantile(loss,.95)),'retention_max':float(loss.max()),'GD_true':GD(reduced,Rn),'IGD_true':IGD(reduced,Rn),'HV':hv,'HV_se':hse,'Spacing':spacing(reduced),'Sparsity':sparsity(reduced)}
            bench.append({'scenario':row.scenario,'seed':int(row.seed),'cluster_method':cluster_method,'cluster_seed':cluster_seed,'n_original':len(Fn),'n_target':target,'n_selected':len(idx),'exact_cardinality':exact,'status':status,'wall_seconds':elapsed,**values})
bench=pd.DataFrame(bench); bench.to_csv(TAB/f'{MODE.lower()}_cardinality_reduction_trials.csv',index=False)
metric_columns=['retention_IGD','retention_p95','retention_max','GD_true','IGD_true','HV','HV_se','Spacing','Sparsity','wall_seconds']
block=bench.groupby(['scenario','seed','cluster_method'],as_index=False).agg(n_trials=('cluster_seed','size'),exact_rate=('exact_cardinality','mean'),**{f'{column}_median':(column,'median') for column in metric_columns},**{f'{column}_iqr':(column,lambda x:x.quantile(.75)-x.quantile(.25)) for column in metric_columns})
global_exact=bench.groupby('cluster_method').exact_cardinality.mean(); eligible=set(global_exact[global_exact.eq(1.0)].index); assert eligible,'Nenhum método preservou a cardinalidade exata'
for metric,ascending in [('IGD_true_median',True),('retention_IGD_median',True),('HV_median',False)]:
    block[f'rank_{metric}']=np.nan; mask=block.cluster_method.isin(eligible); block.loc[mask,f'rank_{metric}']=block[mask].groupby(['scenario','seed'])[metric].rank(method='average',ascending=ascending)
block.to_csv(TAB/f'{MODE.lower()}_cardinality_reduction_block_summary.csv',index=False)
comparison=block.groupby('cluster_method',as_index=False).agg(blocks=('scenario','size'),exact_block_rate=('exact_rate','mean'),median_rank_IGD_true=('rank_IGD_true_median','median'),mean_rank_IGD_true=('rank_IGD_true_median','mean'),IGD_true_median=('IGD_true_median','median'),retention_IGD_median=('retention_IGD_median','median'),retention_p95_median=('retention_p95_median','median'),HV_median=('HV_median','median'),Spacing_median=('Spacing_median','median'),Sparsity_median=('Sparsity_median','median'),wall_seconds_median=('wall_seconds_median','median'))
comparison['exact_trial_rate']=comparison.cluster_method.map(global_exact); comparison['eligible']=comparison.cluster_method.isin(eligible); comparison['wins_IGD']=comparison.cluster_method.map(block[block.rank_IGD_true_median.eq(1)].cluster_method.value_counts()).fillna(0).astype(int)
comparison=comparison.sort_values(['eligible','median_rank_IGD_true','IGD_true_median','retention_IGD_median','wall_seconds_median'],ascending=[False,True,True,True,True]); winner=str(comparison.iloc[0].cluster_method); assert comparison.iloc[0].eligible
comparison.to_csv(TAB/f'{MODE.lower()}_cardinality_reduction_comparison.csv',index=False)
selection={'mode':MODE,'winner':winner,'winner_label':CLUSTER_LABEL[winner],'selection_rule':'exact cardinality in all trials; minimum median blockwise rank of true IGD; tie-break by median true IGD, retention IGD, then runtime','benchmark_scope':'90 complete CNBI fronts' if MODE=='FULL' else 'available complete CNBI fronts','stochastic_cluster_seeds':CLUSTER_SEEDS,'final_reduction_seed':FINAL_REDUCTION_SEED,'benchmark_hv_samples':BENCH_HV_SAMPLES,'methods':CLUSTER_METHODS}
(TAB/f'{MODE.lower()}_cardinality_reduction_selection.json').write_text(json.dumps(selection,indent=2,ensure_ascii=False),encoding='utf-8')
print('Método de clusterização selecionado:',CLUSTER_LABEL[winner]); print(comparison.to_string(index=False))


In [ ]:
labels=[CLUSTER_LABEL[m].replace(' + representante real','').replace('Farthest-Point Sampling','Farthest-Point\nSampling').replace('MiniBatch K-Means','MiniBatch\nK-Means').replace('Hierárquica Ward','Hierárquica\nWard') for m in CLUSTER_METHODS]
fig,axs=plt.subplots(2,2,figsize=(13.5,9.2),layout='constrained')
panels=[('IGD_true_median','IGD contra a fronteira verdadeira',True),('retention_IGD_median','Perda de cobertura da frente CNBI original',True),('rank_IGD_true_median','Posto da IGD por cenário–semente',False),('wall_seconds_median','Tempo de redução por frente (s)',True)]
for ax,(column,title,logscale) in zip(axs.ravel(),panels):
    values=[block.loc[block.cluster_method.eq(method),column].dropna().to_numpy() for method in CLUSTER_METHODS]; box=ax.boxplot(values,labels=labels,patch_artist=True,showfliers=False,medianprops={'color':'#222222','linewidth':1.4})
    for patch,method in zip(box['boxes'],CLUSTER_METHODS): patch.set_facecolor(CLUSTER_COLOR[method]); patch.set_alpha(.72)
    ax.set_title(title); ax.grid(axis='y',alpha=.18); ax.tick_params(axis='x',labelrotation=8)
    if logscale and all(np.all(v>0) for v in values if len(v)): ax.set_yscale('log')
fig.suptitle('Comparação dos métodos de redução de cardinalidade do CNBI')
png=RED_FIG/f'{MODE.lower()}_cardinality_reduction_comparison.png'; pdf=RED_FIG/f'{MODE.lower()}_cardinality_reduction_comparison.pdf'; fig.savefig(png,dpi=300,bbox_inches='tight'); fig.savefig(pdf,dpi=300,bbox_inches='tight'); plt.close(fig)


In [ ]:
def paired_stats(df,metric):
    wide=df.pivot(index=['scenario','seed'],columns='method',values=metric).dropna(); methods=list(wide)
    if len(methods)<3 or len(wide)<3: return pd.DataFrame()
    friedman=stats.friedmanchisquare(*[wide[m] for m in methods]); rows=[]
    for method in methods:
        if method=='CNBI': continue
        difference=(wide['CNBI']-wide[method]) if metric=='HV' else (wide[method]-wide['CNBI']); nonzero=difference[difference!=0]
        if len(nonzero): test=stats.wilcoxon(difference,zero_method='wilcox'); pvalue=float(test.pvalue); ranks=stats.rankdata(np.abs(nonzero)); total=float(ranks.sum()); effect=float((ranks[nonzero>0].sum()-ranks[nonzero<0].sum())/total) if total else 0.0
        else: pvalue=1.0; effect=0.0
        rows.append({'method':method,'p_raw':pvalue,'effect_rank_biserial':effect,'effect_positive_favors':'CNBI','zero_differences':int((difference==0).sum())})
    out=pd.DataFrame(rows); out['p_holm']=multipletests(out.p_raw,method='holm')[1]; out['friedman_p']=friedman.pvalue; return out

equal_rows=[]
for (scenario,campaign_seed),group in records.groupby(['scenario','seed'],sort=True):
    target=targets[(scenario,int(campaign_seed))]; m=int(scenario.split('_')[0][1:]); Z=qmc.Sobol(m,scramble=True,seed=2026+int(campaign_seed)).random(int(CFG['hv_samples']))*1.1
    for row in group.itertuples(index=False):
        _,Fn,_,Rn,_=load_front(row); idx=reduce_indices(Fn,target,winner,FINAL_REDUCTION_SEED) if len(Fn)>target else np.arange(len(Fn)); assert len(idx)==target
        reduced=Fn[idx]; hv,hse=hv_from_samples(reduced,Z); base=complete[(complete.scenario==scenario)&(complete.seed.astype(int)==int(campaign_seed))&(complete.method==row.method)].iloc[0].to_dict()
        base.update({'comparison':'equal_cardinality','n':len(reduced),'GD':GD(reduced,Rn),'IGD':IGD(reduced,Rn),'HV':hv,'HV_se':hse,'Spacing':spacing(reduced),'Sparsity':sparsity(reduced),'equalization_method':winner,'equalization_seed':FINAL_REDUCTION_SEED}); equal_rows.append(base)
complete_out=complete.copy(); complete_out['equalization_method']='not_applicable'; complete_out['equalization_seed']=np.nan
metrics_out=pd.concat([complete_out,pd.DataFrame(equal_rows)],ignore_index=True); metrics_out.to_csv(TAB/f'{MODE.lower()}_metrics.csv',index=False)
summary=metrics_out.groupby(['scenario','method','comparison'],as_index=False).agg(**{f'{metric}_median':(metric,'median') for metric in ('GD','IGD','HV','Spacing','Sparsity')},**{f'{metric}_iqr':(metric,lambda x:x.quantile(.75)-x.quantile(.25)) for metric in ('GD','IGD','HV','Spacing','Sparsity')}); summary.to_csv(TAB/f'{MODE.lower()}_summary.csv',index=False)
rank_rows=[]
for (scenario,comparison_name,campaign_seed),group in metrics_out.groupby(['scenario','comparison','seed']):
    for metric in ('GD','IGD','HV','Spacing','Sparsity'):
        ranks=group[metric].rank(method='average',ascending=metric!='HV')
        for method,rank in zip(group.method,ranks): rank_rows.append({'scenario':scenario,'comparison':comparison_name,'seed':campaign_seed,'metric':metric,'method':method,'rank':rank})
pd.DataFrame(rank_rows).to_csv(TAB/f'{MODE.lower()}_rankings.csv',index=False)
stats_rows=[]
for (scenario,comparison_name),group in metrics_out.groupby(['scenario','comparison']):
    for metric in ('GD','IGD','HV','Spacing','Sparsity'):
        if group.seed.nunique()<3: stats_rows.append({'scenario':scenario,'comparison':comparison_name,'metric':metric,'test':'Friedman/Wilcoxon-Holm','status':'INSUFFICIENT_BLOCKS','n_blocks':group.seed.nunique()}); continue
        result=paired_stats(group,metric)
        if result.empty: stats_rows.append({'scenario':scenario,'comparison':comparison_name,'metric':metric,'test':'Friedman/Wilcoxon-Holm','status':'NO_VALID_COMMON_BLOCK','n_blocks':group.seed.nunique()})
        else:
            result.insert(0,'status','COMPLETED'); result.insert(0,'metric',metric); result.insert(0,'comparison',comparison_name); result.insert(0,'scenario',scenario); stats_rows.extend(result.to_dict('records'))
pd.DataFrame(stats_rows).to_csv(TAB/f'{MODE.lower()}_statistics.csv',index=False)
print(f'Comparações equalizadas recalculadas com {winner}: {len(equal_rows)} linhas.')


In [ ]:
def plot_handles(methods):
    items=[Line2D([0],[0],marker='o',linestyle='none',markerfacecolor='#a7adb4',markeredgecolor='none',alpha=.55,label='fronteira verdadeira')]
    items.extend(Line2D([0],[0],marker=METHOD_MARKER[m],linestyle='none',markerfacecolor=METHOD_COLOR[m],markeredgecolor='white',markeredgewidth=.4,markersize=6,label=m) for m in methods)
    items.append(Line2D([0],[0],marker='*',linestyle='none',markerfacecolor='white',markeredgecolor='.1',markersize=8,label='ótimos individuais verdadeiros')); return items
def anchor_objectives(anchors): return np.sum((anchors[:,None,:]-anchors[None,:,:])**2,axis=2)
def sample_reference(F,limit,seed):
    if len(F)<=limit: return F
    idx=np.sort(np.random.default_rng(seed).choice(len(F),size=limit,replace=False)); return F[idx]
def draw2d(ax,pair,Rraw,fronts,methods,Fanchors):
    i,j=pair[0]-1,pair[1]-1; reference=sample_reference(Rraw,6000,1907+i*31+j); ax.scatter(reference[:,i],reference[:,j],s=4,c='#a7adb4',alpha=.18,linewidths=0,rasterized=True)
    for method in methods:
        F=fronts[method]; ax.scatter(F[:,i],F[:,j],s=25,c=METHOD_COLOR[method],marker=METHOD_MARKER[method],alpha=.78,edgecolors='white',linewidths=.35,rasterized=True)
    ax.scatter(Fanchors[:,i],Fanchors[:,j],marker='*',s=55,c='white',edgecolors='.1',linewidths=.7,zorder=8); ax.set(xlabel=rf'$f_{{{i+1}}}$',ylabel=rf'$f_{{{j+1}}}$',title=rf'Projeção $f_{{{i+1}}}\times f_{{{j+1}}}$'); ax.grid(alpha=.15)
def draw3d(ax,triple,Rraw,fronts,methods,Fanchors):
    i,j,k=triple[0]-1,triple[1]-1,triple[2]-1; reference=sample_reference(Rraw,4500,2909+i*31+j*17+k); ax.scatter(reference[:,i],reference[:,j],reference[:,k],s=3,c='#a7adb4',alpha=.12,linewidths=0,depthshade=False,rasterized=True)
    for method in methods:
        F=fronts[method]; ax.scatter(F[:,i],F[:,j],F[:,k],s=24,c=METHOD_COLOR[method],marker=METHOD_MARKER[method],alpha=.82,edgecolors='white',linewidths=.35,depthshade=False,rasterized=True)
    ax.scatter(Fanchors[:,i],Fanchors[:,j],Fanchors[:,k],marker='*',s=58,c='white',edgecolors='.1',linewidths=.7,depthshade=False); ax.set(xlabel=rf'$f_{{{i+1}}}$',ylabel=rf'$f_{{{j+1}}}$',title=rf'Projeção $(f_{{{i+1}}},f_{{{j+1}}},f_{{{k+1}}})$'); ax.set_zlabel(''); ax.text2D(.91,.52,rf'$f_{{{k+1}}}$',transform=ax.transAxes,rotation=90,va='center',ha='center'); ax.view_init(elev=ISO_ELEV,azim=ISO_AZIM); ax.set_proj_type('ortho'); ax.set_box_aspect((1,1,1)); ax.grid(alpha=.15)
    for axis in (ax.xaxis,ax.yaxis,ax.zaxis): axis.pane.set_facecolor((1,1,1,0))
def save_plate(scenario,projections,dimension,Rraw,fronts,methods,Fanchors,target,out_dir):
    n=len(projections); fig=plt.figure(figsize=(6.1*n,6.2 if dimension=='2D' else 6.8),layout='constrained')
    for position,projection in enumerate(projections,1):
        ax=fig.add_subplot(1,n,position,projection=None if dimension=='2D' else '3d'); (draw2d if dimension=='2D' else draw3d)(ax,projection,Rraw,fronts,methods,Fanchors)
    fig.legend(handles=plot_handles(methods),loc='outside lower center',ncol=len(methods)+2,frameon=False); suffix='2d' if dimension=='2D' else '3d_isometric'; fig.suptitle(f'{scenario}: cardinalidade igual n={target}, semente comum {VISUAL_SEED}' + ('' if dimension=='2D' else ' — vistas isométricas'))
    png=out_dir/f'{scenario}_equal_cardinality_plate_{suffix}.png'; pdf=out_dir/f'{scenario}_equal_cardinality_plate_{suffix}.pdf'; fig.savefig(png,dpi=300,bbox_inches='tight'); fig.savefig(pdf,dpi=300,bbox_inches='tight'); plt.close(fig); return png,pdf

figure_rows=[]
for scenario,spec in SELECTIONS.items():
    group=records[(records.scenario==scenario)&(records.seed.astype(int)==VISUAL_SEED)].copy(); target=targets[(scenario,VISUAL_SEED)]; fronts={}; Rraw=None; anchors=None
    for row in group.itertuples(index=False):
        Fraw,Fn,current_reference,_,current_anchors=load_front(row); idx=reduce_indices(Fn,target,winner,FINAL_REDUCTION_SEED) if len(Fn)>target else np.arange(len(Fn)); assert len(idx)==target; fronts[row.method]=Fraw[idx]; Rraw=current_reference; anchors=current_anchors
    methods=[method for method in METHOD_ORDER if method in fronts]; out_dir=EQ_FIG/scenario; out_dir.mkdir(parents=True,exist_ok=True); Fanchors=anchor_objectives(anchors)
    png2,pdf2=save_plate(scenario,spec['pairs'],'2D',Rraw,fronts,methods,Fanchors,target,out_dir); png3,pdf3=save_plate(scenario,spec['triples'],'3D',Rraw,fronts,methods,Fanchors,target,out_dir)
    for dimension,projections,png,pdf in [('2D',spec['pairs'],png2,pdf2),('3D',spec['triples'],png3,pdf3)]:
        for projection in projections: figure_rows.append({'scenario':scenario,'dimension':dimension,'projection':'-'.join(map(str,projection)),'seed':VISUAL_SEED,'target_n':target,'cluster_method':winner,'methods':'|'.join(methods),'plate_png':png.relative_to(ROOT).as_posix(),'plate_pdf':pdf.relative_to(ROOT).as_posix(),'view_elev':ISO_ELEV if dimension=='3D' else np.nan,'view_azim':ISO_AZIM if dimension=='3D' else np.nan})
figure_manifest=pd.DataFrame(figure_rows); figure_manifest.to_csv(EQ_FIG/f'{MODE.lower()}_equalized_projection_manifest.csv',index=False)
metadata={'mode':MODE,'cluster_method':winner,'cluster_method_label':CLUSTER_LABEL[winner],'reduction_seed':FINAL_REDUCTION_SEED,'visual_comparison_seed':VISUAL_SEED,'cardinality_target':'minimum valid complete-front cardinality within each scenario-seed block','method_colors':METHOD_COLOR,'method_markers':METHOD_MARKER,'three_dimensional_view':{'projection':'orthographic','elevation_degrees':ISO_ELEV,'azimuth_degrees':ISO_AZIM}}; (EQ_FIG/f'{MODE.lower()}_equalized_projection_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print(f'Figuras equalizadas: {len(figure_manifest)} projeções em {figure_manifest.scenario.nunique()} cenários.')
